# 02 - Preprocesado

**Proyecto Deep Learning: Detección de Imágenes Generadas por IA (CIFAKE)**

En este notebook se prepara el dato para el entrenamiento: **normalización** de los tensores al rango [0, 1] (división por 255.0) y configuración de la canalización de datos con **`cache()`** y **`prefetch()`** para acelerar la lectura.

## 1. Instalar e importar librerías

In [1]:
!pip install -q kagglehub

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f'TensorFlow: {tf.__version__}')

TensorFlow: 2.21.0


## 2. Descargar y cargar los datos

Se repite la ingesta para que el notebook sea autónomo en Colab.

In [3]:
import kagglehub

# Descarga el dataset CIFAKE desde Kaggle (queda en cache la primera vez)
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print('Ruta del dataset:', path)

Ruta del dataset: C:\Users\Anonymous\.cache\kagglehub\datasets\birdy654\cifake-real-and-ai-generated-synthetic-images\versions\3


In [4]:
train_path = Path(path) / 'train'
test_path = Path(path) / 'test'

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_path, image_size=(32, 32), batch_size=32,
    class_names=['FAKE', 'REAL'], label_mode='binary')

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_path, image_size=(32, 32), batch_size=32,
    class_names=['FAKE', 'REAL'], label_mode='binary')

print('Datos cargados')

Found 100000 files belonging to 2 classes.


Found 20000 files belonging to 2 classes.


Datos cargados


## 3. Normalización y canalización de datos

- **Normalización:** cada pixel se divide por 255.0 para llevarlo del rango [0, 255] al rango [0, 1]. Esto estabiliza y acelera el entrenamiento de la red.
- **`cache()`:** mantiene los datos en memoria tras la primera época.
- **`prefetch(AUTOTUNE)`:** prepara el siguiente lote mientras la GPU/CPU procesa el actual, reduciendo cuellos de botella de E/S.

In [5]:
def normalize(images, labels):
    return images / 255.0, labels

train_ds = train_ds.map(normalize).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(normalize).cache().prefetch(tf.data.AUTOTUNE)

print('Datos normalizados')

Datos normalizados


## 4. Verificar el preprocesado

Se comprueba que los valores de los píxeles quedaron efectivamente en el rango [0, 1].

In [6]:
for images, labels in train_ds.take(1):
    print('Forma del lote de imágenes:', images.shape)
    print('Valor mínimo:', float(tf.reduce_min(images)))
    print('Valor máximo:', float(tf.reduce_max(images)))
    print('Forma de las etiquetas:', labels.shape)
    break

Forma del lote de imágenes:

 (32, 32, 32, 3)
Valor mínimo: 0.0
Valor máximo: 1.0
Forma de las etiquetas: (32, 1)


### Resultado esperado

El valor mínimo debe ser cercano a `0.0` y el máximo cercano a `1.0`, confirmando que la normalización fue aplicada correctamente. Los datos quedan listos para alimentar la CNN del notebook `03`.